
# 02 · Chiến lược — kiến trúc 8 agents

**Vai trò notebook**: walk through từng agent — kiến trúc, code reference, 1 sample decision.

**Owner**: Duc
**Deadline**: 2026-05-22
**Slide chapter**: 4 — Implementation (Strategies section)

## Mục tiêu
Cho mỗi agent (8 total) một section ngắn:
1. **Một câu** mô tả nó là gì
2. **Code reference** — file path + function chính
3. **Hyper-parameters** (locked, ref PRD §15)
4. **One sample decision** — input state → output target weights

## Defense Q&A
- Q: Tại sao DDPG saturated tanh? Backup PPO khắc phục thế nào?
- Q: Multi-agent có 8 roles — vai trò mỗi role là gì?
- Q: LLM weekly vs DDPG daily — công bằng không khi so sánh?
- Q: Sao không dùng SAC/TD3 thay DDPG?

## Order trình bày
Đi từ đơn giản → phức tạp:
buy_and_hold → equal_weight → random → DDPG → PPO → zero_shot → single_agentic → multi_agent


## Setup


In [ ]:
import sys
from pathlib import Path

# Make `from _shared import ...` work whether you run from notebooks/ or repo root.
_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: arch-overview — figure 1 tổng quan 8 agents

- **OWNER**: Duc   **DEPENDS**: none
- **WRITE**: `report/figures/02__arch_overview.png`
- **CONSTRAINTS**:
  - 1 figure với 3 columns (Baselines / RL / LLM)
  - Mỗi column liệt kê agents trong nhóm + 1-line description
  - Color theo AGENT_COLORS
  - Title VI: "Tổng quan 8 chiến lược — baselines, RL, LLM"
- **VALIDATE**: file exists, ≥ 50KB
- **PATTERN**: matplotlib `text()` + `Rectangle` patches; hoặc draw manual với plt.annotate
- **DEFENSE Q&A**: "Có bao nhiêu agents trong project và phân loại như nào?" → mở figure này


In [ ]:
# TODO-01: 3-column architecture overview
fig, ax = plt.subplots(figsize=(12, 6))
# ...
save_fig('02__arch_overview')



## TODO-02: baselines-section — buy_and_hold + equal_weight + random
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/baselines.py`
- **WRITE**: 3 markdown subsections + 1 code cell mỗi agent showing decide() logic
- **CONSTRAINTS**:
  Mỗi subsection format:
  ```
  ### buy_and_hold
  - One-liner: "Mua đều 5 ticker ngày đầu test, giữ nguyên đến hết"
  - File: src/baselines.py:BuyAndHold
  - Action shape: 1/N (uniform) at t=0, then 0 (no rebalance)
  - Sample: in ra `BuyAndHold().decide(state_t0)` cho 5 ticker
  ```
- **VALIDATE**: 3 sections render, mỗi cái có sample output
- **DEFENSE Q&A**: "Tại sao bao gồm random agent?" → để định nghĩa floor; agent thực phải beat random.


In [ ]:
# TODO-02: baselines decide() samples
from src.baselines import BuyAndHold, EqualWeightRebalance, RandomAgent
# ...



## TODO-03: rl-section — DDPG + PPO architecture
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/ddpg_trainer.py`, `src/ppo_trainer.py`, `src/agents/rl_agent.py`
- **WRITE**:
  - Markdown: actor-critic explanation (DDPG continuous action, PPO clipped objective)
  - `report/figures/02__ddpg_ppo_arch.png` — diagram actor + critic networks
  - Code cell: load trained model, run 1 forward pass with state mẫu
- **CONSTRAINTS**:
  - Network shape từ trainer config (hidden_dim, layers)
  - State dim = 5 tickers × (n_features) — ref `VNTradingEnv.observation_space`
  - Action dim = 5 (target weight mỗi ticker, range [-1, 1])
  - **GOTCHA**: DDPG saturated tanh (PRD §14 Risk #7) — cần note rõ trong markdown
- **VALIDATE**:
  - `assert action.shape == (5,)`
  - `assert action.min() >= -1 and action.max() <= 1`
- **PATTERN**: `stable_baselines3.DDPG.load(...).predict(obs)`
- **DEFENSE Q&A**: "Tại sao PPO khắc phục được vấn đề DDPG?" → clipped surrogate objective + entropy bonus prevents premature convergence.


In [ ]:
# TODO-03: DDPG + PPO sample forward pass
from stable_baselines3 import DDPG, PPO
# ...



## TODO-04: zero-shot-section — LLM zero-shot
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/llm/zero_shot.py`, `src/llm/serialize.py`, `src/llm/client.py`
- **WRITE**:
  - Markdown: prompt template summary, model lock (gpt-4o-mini), no tools, no debate
  - Code cell: load 1 cached zero_shot decision, parse weights, print
- **CONSTRAINTS**:
  - Show prompt template excerpt (10-20 lines, không dùng full vì dài)
  - Show 1 sample (state, prompt, response, parsed_weights)
- **VALIDATE**: parsed weights length == 5, sum ≤ 1
- **PATTERN**: `results/zero_shot/decisions.jsonl` chứa cached responses
- **DEFENSE Q&A**: "Zero-shot có lợi thế gì vs RL?" → no training data needed, leverages pretrained knowledge.


In [ ]:
# TODO-04: zero_shot decision sample
import json
sample_path = RESULTS / 'zero_shot' / 'decisions.jsonl'
# ...



## TODO-05: single-agentic-section — single LLM with tools
- **OWNER**: Duc   **DEPENDS**: TODO-04
- **READ**: `src/llm/single_agentic.py`, `src/llm/tools.py`
- **WRITE**:
  - Markdown: 4 tools (get_price, get_indicators, get_news, get_fundamentals) + ReAct loop
  - `report/figures/02__single_agentic_loop.png` — diagram tool-call cycle
  - Code cell: show 1 cached transcript (LLM nghĩ → call tool → tool result → continue)
- **CONSTRAINTS**:
  - Diagram: LLM (gpt-4o) → tool[?] → state → LLM → ...
  - Max iterations = 5 (config)
- **VALIDATE**: figure exists; sample transcript has ≥ 1 tool call
- **PATTERN**: `results/single_agentic/decisions.jsonl` for cached transcripts
- **DEFENSE Q&A**: "ReAct vs Tree-of-Thought?" → chose ReAct cho minimal iteration count + cost control.


In [ ]:
# TODO-05: single_agentic loop diagram + sample
pass



## TODO-06: multi-agent-topology — 8-role debate graph
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/llm/multi_agent/graph.py`, `src/llm/multi_agent/state.py`, sample transcript
- **WRITE**:
  - `report/figures/02__multi_agent_topology.png` — 8 nodes + edges (mirror PKG-S frontend graph)
  - Markdown: bảng 8 roles × (model, primary output, downstream consumer)
- **CONSTRAINTS**:
  - Topology: 3 analysts (technical/news/fundamental) → bull/bear debate 2 rounds → trader → risk → portfolio
  - Models: gpt-4o-mini for analysts; gpt-4o for researchers/trader/risk/portfolio
  - Use ROLE_COLORS for node fill
- **VALIDATE**:
  - 8 nodes rendered
  - All edges match `graph.py` topology
- **PATTERN**: tham khảo `frontend/components/DebateGraph.tsx` cho layout positions
- **DEFENSE Q&A** (critical): "Vẽ kiến trúc multi-agent của bạn?" → mở figure này


In [ ]:
# TODO-06: multi-agent topology diagram
pass



## TODO-07: multi-agent-sample — replay 1 decision đầy đủ
- **OWNER**: Duc   **DEPENDS**: TODO-06
- **READ**: `results/multi_agent/transcripts/2026-04-28.json` (1 sample ngày gần cuối test)
- **WRITE**: 8 markdown cells (1 per role) showing role + output excerpt (200 chars)
- **CONSTRAINTS**:
  - For each role in topological order:
    ```
    ### technical_analyst (gpt-4o-mini)
    > "## VCB\nSetup: sideways... Nhận định: **neutral** — chưa có tín hiệu mạnh."
    ```
  - Final: portfolio_manager's full JSON decision (weights)
- **VALIDATE**: 10 transcript entries rendered (3 analysts + 2 bull + 2 bear + trader + risk + portfolio)
- **DEFENSE Q&A**: "Cho xem ví dụ 1 quyết định của multi-agent?" → walk qua 10 cells của notebook này


In [ ]:
# TODO-07: replay 2026-04-28 decision
transcript = load_transcript('2026-04-28')
# ...



## TODO-08: cadence-comparison — RL daily vs LLM weekly
- **OWNER**: Duc   **DEPENDS**: none
- **WRITE**: `report/figures/02__cadence.png` — bar chart n_steps + n_decisions
- **CONSTRAINTS**:
  - Bar group 1: n_steps (247 cho all daily agents)
  - Bar group 2: n_decisions (chỉ LLM agents: zero_shot=10 smoke, single=10 smoke, multi=51 full)
  - Markdown explanation: tại sao LLM weekly = cost control ($0.05/run × daily × 248 = $12 quá đắt)
- **VALIDATE**: figure exists; both bar groups present
- **DEFENSE Q&A**: "So sánh daily vs weekly có công bằng không?" → có; vì env handle rebalance daily, weighting daily nhưng decision LLM chỉ tới mỗi tuần. RL/LLM share same execution layer.


In [ ]:
# TODO-08: decision cadence bar
pass



## Defense Q&A — câu trả lời sẵn

> **Q1: Tại sao DDPG saturate tanh? PPO khắc phục thế nào?**
> A: DDPG deterministic policy + Ornstein-Uhlenbeck noise → action gradient nhỏ khi tanh ≈ ±1 → policy stuck overweight 1 ticker. PPO clipped surrogate + entropy bonus duy trì exploration; ngoài ra PPO stochastic policy nên tránh deterministic boundary.
> Evidence: `results/{ddpg,ppo}_training_log.jsonl` actor loss curves + PRD §14 Risk #7.

> **Q2: 8 roles trong multi-agent làm gì?**
> A: 3 analysts (technical/news/fundamental) cung cấp signal độc lập. Bull/bear researcher tranh luận 2 vòng để stress-test conviction. Trader tổng hợp thành proposal. Risk manager review concentration + drawdown. Portfolio manager finalize weights JSON.
> Evidence: TODO-06 topology + TODO-07 sample replay.

> **Q3: LLM weekly vs RL daily công bằng không?**
> A: Có. Env execution layer giống nhau (daily rebalance), chỉ decision frequency khác. Comparison metric (cumulative_return, sharpe) tính trên daily PnL của cả 2 nhóm. Cost rationale: LLM 51 calls/test ≈ $3 vs daily LLM 248 calls ≈ $15.
> Evidence: TODO-08 + `results/multi_agent/metrics.json` field n_decisions.

> **Q4: Tại sao không dùng SAC/TD3?**
> A: Out of scope — PRD §4 lock DDPG (paper Xiong et al) + PPO (backup khi DDPG saturate). SAC/TD3 explore là post-MVP work.
